<a href="https://colab.research.google.com/github/soominlee0726/Email_Generator_for_students/blob/main/Experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Qwen2.5-1.5B-Instruct - zero shot

In [ ]:
!pip -q install "transformers>=4.37.0" accelerate sentencepiece

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

In [ ]:
def generate_email(user_input):
    system_prompt = """
너는 대학생을 도와 교수님께 보내는 정중한 한국어 이메일을 작성하는 assistant다.
항상 다음 원칙을 지켜라.

1. 한국어로만 작성한다.
2. 공손하고 자연스러운 메일 문체를 사용한다.
3. 메일 구조는 다음 흐름을 따른다:
   - 인사
   - 자기소개
   - 상황 설명
   - 요청 또는 문의
   - 마무리
4. 과장하거나 거짓 정보를 추가하지 않는다.
5. 출력은 이메일 본문만 작성한다.
6. 학생 이름은 'OOO'으로 쓴다.
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=300,
        do_sample=False,
        num_beams=4
    )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

In [ ]:
test_input = "이수민 교수님께 자연어처리 수업 출결 관련 메일 작성. 다음 주 병원 진료로 인해 수업에 참석하기 어렵습니다."
print(generate_email(test_input))

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:509: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


이수민 교수님께 보내는 이메일입니다.

안녕하세요, 이수민 교수님. 저는 이수민 교수님의 자연어처리 수업에 참석하려고 노력했지만, 다음 주 병원 진료로 인해 수업에 참석하기 어려운 상황에 처하게 되었습니다. 이에 대해 깊은 죄송함을 느끼며, 수업에 참석할 수 있도록 최선을 다하겠습니다.

감사합니다.
OOO


## pki-t5-small Model    
The experimental results for the FLAN-T5-small model were excluded due to compatibility issues arising from library version conflicts in the transformer framework, which prevented stable reproduction of the training pipeline.

In [ ]:
!pip -q install "transformers==4.40.2" "datasets==2.19.1" "accelerate==0.29.3" "peft==0.10.0" sentencepiece

In [ ]:
import transformers, datasets, peft, accelerate

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)

assert transformers.__version__ == "4.40.2", "transformers 버전이 다릅니다. 설치 후 런타임 재시작하고 다시 실행하세요."
assert datasets.__version__ == "2.19.1", "datasets 버전이 다릅니다. 설치 후 런타임 재시작하고 다시 실행하세요."
assert peft.__version__ == "0.10.0", "peft 버전이 다릅니다. 설치 후 런타임 재시작하고 다시 실행하세요."
assert accelerate.__version__ == "0.29.3", "accelerate 버전이 다릅니다. 설치 후 런타임 재시작하고 다시 실행하세요."

print("버전 검증 통과")

transformers: 4.40.2
datasets: 2.19.1
peft: 0.10.0
accelerate: 0.29.3
버전 검증 통과


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

JSONL_PATH = "/content/drive/MyDrive/NLP project/train.jsonl"
assert os.path.exists(JSONL_PATH), f"파일을 못 찾음: {JSONL_PATH}"
print("파일 확인:", JSONL_PATH)

In [ ]:
{"input":"이수민 교수님께 자연어처리 수업 출결 관련 메일 작성. 다음 주 몸살로 인해 수업에 참석하기 어려울 것 같습니다.","output":"안녕하세요, 이수민 교수님...\nOOO 드림"}
{"input":"박정우 교수님께 회로이론 수업 과제 제출 연장 관련 메일 작성. ...","output":"안녕하세요, 박정우 교수님...\nOOO 드림"}

{'input': '박정우 교수님께 회로이론 수업 과제 제출 연장 관련 메일 작성. ...',
 'output': '안녕하세요, 박정우 교수님...\nOOO 드림'}

In [ ]:
import json
from pathlib import Path

def validate_jsonl(path):
    path = Path(path)
    assert path.exists(), f"파일이 없습니다: {path}"

    rows = []
    bad = []

    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            raw = line.rstrip("\n")

            if not raw.strip():
                bad.append((line_no, "빈 줄"))
                continue

            try:
                obj = json.loads(raw)
            except Exception as e:
                bad.append((line_no, f"JSON 파싱 실패: {e}"))
                continue

            if not isinstance(obj, dict):
                bad.append((line_no, "JSON 객체(dict)가 아님"))
                continue

            missing = [k for k in ["input", "output"] if k not in obj]
            if missing:
                bad.append((line_no, f"키 누락: {missing}"))
                continue

            if not isinstance(obj["input"], str) or not isinstance(obj["output"], str):
                bad.append((line_no, "input/output이 문자열이 아님"))
                continue

            if not obj["input"].strip() or not obj["output"].strip():
                bad.append((line_no, "input 또는 output이 비어 있음"))
                continue

            rows.append({
                "input": obj["input"].strip(),
                "output": obj["output"].strip()
            })

    return rows, bad

rows, bad = validate_jsonl(JSONL_PATH)

print(f"정상 샘플 수: {len(rows)}")
print(f"문제 있는 줄 수: {len(bad)}")

if bad:
    print("\n문제 있는 줄 예시:")
    for item in bad[:10]:
        print(item)
    raise ValueError("train.jsonl 형식에 문제가 있습니다. 위 줄 번호를 수정한 뒤 다시 실행하세요.")

print("\n첫 샘플 input:")
print(rows[0]["input"])
print("\n첫 샘플 output:")
print(rows[0]["output"][:300])

정상 샘플 수: 40
문제 있는 줄 수: 0

첫 샘플 input:
이수민 교수님께 자연어처리 수업 출결 관련 메일 작성. 다음 주 몸살로 인해 수업에 참석하기 어려울 것 같습니다.

첫 샘플 output:
안녕하세요, 이수민 교수님.
저는 교수님의 자연어처리 수업을 수강 중인 OOO입니다.

다음 주 몸살 증상으로 인해 부득이하게 수업에 참석하기 어려울 것 같아 메일 드립니다. 갑작스럽게 연락드리게 되어 죄송합니다. 혹시 출결과 관련하여 제가 추가로 확인하거나 제출해야 할 사항이 있다면 알려주시면 감사하겠습니다.

감사합니다.
OOO 드림


In [ ]:
from collections import Counter

inputs = [r["input"] for r in rows]
outputs = [r["output"] for r in rows]

print("입력 평균 길이:", sum(len(x) for x in inputs) / len(inputs))
print("출력 평균 길이:", sum(len(x) for x in outputs) / len(outputs))
print("입력 최소/최대 길이:", min(len(x) for x in inputs), max(len(x) for x in inputs))
print("출력 최소/최대 길이:", min(len(x) for x in outputs), max(len(x) for x in outputs))

dup_inputs = [k for k, v in Counter(inputs).items() if v > 1]
print("중복 input 개수:", len(dup_inputs))

too_short = [i for i, r in enumerate(rows) if len(r["output"]) < 30]
print("너무 짧은 output 샘플 개수:", len(too_short))

입력 평균 길이: 70.3
출력 평균 길이: 190.8
입력 최소/최대 길이: 57 88
출력 최소/최대 길이: 162 214
중복 input 개수: 15
너무 짧은 output 샘플 개수: 0


In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(rows)
dataset = dataset.shuffle(seed=42)

print(dataset)
print(dataset[0])

In [ ]:
from transformers import T5TokenizerFast, T5ForConditionalGeneration

MODEL_NAME = "paust/pko-t5-small"

tokenizer = T5TokenizerFast.from_pretrained(MODEL_NAME)
base_model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

print("모델/토크나이저 로드 완료")

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

In [ ]:
MAX_SOURCE_LEN = 128
MAX_TARGET_LEN = 256
PREFIX = "교수님께 보낼 정중한 이메일 작성: "

def preprocess_function(batch):
    inputs = [PREFIX + x for x in batch["input"]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LEN,
        truncation=True,
    )

    labels = tokenizer(
        text_target=batch["output"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset.column_names
)

print(tokenized_dataset)
print(tokenized_dataset[0].keys())

In [ ]:
input_lens = [len(x) for x in tokenized_dataset["input_ids"]]
label_lens = [len(x) for x in tokenized_dataset["labels"]]

print("입력 토큰 평균 길이:", sum(input_lens) / len(input_lens))
print("출력 토큰 평균 길이:", sum(label_lens) / len(label_lens))
print("입력 토큰 최소/최대:", min(input_lens), max(input_lens))
print("출력 토큰 최소/최대:", min(label_lens), max(label_lens))

sample_idx = 0
print("\n[디코딩된 입력]")
print(tokenizer.decode(tokenized_dataset[sample_idx]["input_ids"], skip_special_tokens=True))

print("\n[디코딩된 라벨]")
print(tokenizer.decode([x for x in tokenized_dataset[sample_idx]["labels"] if x != -100], skip_special_tokens=True))

입력 토큰 평균 길이: 60.55
출력 토큰 평균 길이: 122.95
입력 토큰 최소/최대: 48 73
출력 토큰 최소/최대: 105 138

[디코딩된 입력]
교수님께 보낼 정중한 이메일 작성: 오세훈 교수님께 반도체공학 수업 면담 요청 관련 메일 작성. 수업 내용과 연구실 진학에 대해 상담을 받고 싶습니다.

[디코딩된 라벨]
안녕하세요, 오세훈 교수님.
저는 교수님의 반도체공학 수업을 수강 중인 OOO입니다.

반도체공학 수업을 들으면서 연구실 진학에 대한 관심이 생겨 교수님께 조언을 구하고 싶어 메일 드립니다. 가능하시다면 수업 내용과 진로 방향에 대해 짧게라도 면담을 요청드리고 싶습니다. 교수님께서 편하신 시간이 있으시다면 알려주시면 감사하겠습니다.

읽어주셔서 감사합니다.
OOO 드림


In [ ]:
import torch
from transformers import DataCollatorForSeq2Seq

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

sample_batch = [tokenized_dataset[i] for i in range(min(2, len(tokenized_dataset)))]
batch = data_collator(sample_batch)
batch = {k: v.to(device) for k, v in batch.items()}

model.eval()
with torch.no_grad():
    out = model(**batch)

print("드라이런 성공")
print("loss:", float(out.loss))
print({k: tuple(v.shape) for k, v in batch.items()})

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/pko_email_lora",
    num_train_epochs=8,
    per_device_train_batch_size=2,
    learning_rate=1e-4,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_steps=1,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/NLP project/pko_email_lora_adapter"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("저장 완료:", SAVE_DIR)

In [ ]:
model.eval()

def generate_email(user_input, max_new_tokens=220):
    prompt = PREFIX + user_input
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_LEN,
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=4,
        do_sample=False,
        repetition_penalty=1.1,
        no_repeat_ngram_size=3,
        early_stopping=True,
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
test_prompt = "박정우 교수님께 회로이론 수업 과제 제출 연장 관련 메일 작성. 개인적인 사정으로 과제를 기한 내 제출하기 어려워 하루 정도 연장이 가능한지 여쭙고 싶습니다."
print(generate_email(test_prompt))

, 박정우 교수님의 회로이론 수업 과제 제출 연장 관련 메일 부탁드립니다.

안녕하세요. 박정좌 교수 입니다. 문의 주셔서 감사합니다. 답변 부탁드려요. 도와 주시면 감사하겠습니다. 부탁 드립니다. 제가 개인적인 사정으로 인해 과제를 제출하지 못한 상태에서 진행 중인 과제에 대한 문의를 드리고 싶습니다. 빠른 시일 내에 부탁드려도 될까 싶어 글 남깁니다. 질문자님께서 문의주신 사항에 대해 말씀 부탁드리며, 혹시 문의가 있으신 분이 계실까 하여 글을 올립니다. 가능하다면 어떻게 해야 할까요?

문의 주신 내용 중 하나가 있다면 알려주시면 고맙겠습니다. 저는 이 부분에 대해서 궁금한 점이 있어 여쭤봅니다. 
